# **Organização e Download Banco de Dados Áreas Contaminadas CETESB - SP**

Como Citar: GOMES. G, C. Gabriel Caldeira Gomes. Script para Organização e Downaload de Banco de Dados Áreas Contaminadas CETESB - SP. 2026.

In [2]:
!pip install geopandas requests

# **Organização e Download Banco de Dados Áreas Contaminadas CETESB - SP**

Como Citar: GOMES. G, C. Gabriel Caldeira Gomes. Script para Organização e Downaload de Banco de Dados Áreas Contaminadas CETESB - SP. 2026.

In [3]:
import requests
import geopandas as gpd

def baixar_cetesb_arcgis(output_filename="areas_contaminadas_cetesb.gpkg", layer_id=1):
    """
    layer_id=1 : Áreas Contaminadas - Pontos (Geral)
    layer_id=2 : Áreas Contaminadas - Polígonos (Geral)
    """
    print(f"Iniciando o download da camada {layer_id} via ArcGIS REST API...")

    # URL do MapServer atual utilizado pelo portal da SEMIL
    base_url = f"https://mapas.semil.sp.gov.br/server/rest/services/SIGAM/Empreendimento_Contaminacao_SGP/MapServer/{layer_id}/query"

    all_features = []
    offset = 0

    while True:
        params = {
            'where': '1=1', # Traz todos os registros
            'outFields': '*', # Traz todas as colunas
            'f': 'geojson', # Solicita formato GeoJSON
            'resultOffset': offset # Paginação
        }

        response = requests.get(base_url, params=params, timeout=120)
        response.raise_for_status()
        data = response.json()

        # Extrai as feições desta requisição
        features = data.get('features', [])
        if not features:
            break

        all_features.extend(features)
        print(f"Baixados {len(all_features)} registros...")

        # O ArcGIS retorna 'exceededTransferLimit': true se houver mais dados para baixar
        if data.get('exceededTransferLimit'):
            offset += len(features) # Avança para a próxima página
        else:
            break

    if not all_features:
        print("Nenhum dado retornado do servidor.")
        return None

    print("\nProcessando os dados no GeoPandas e convertendo para GPKG...")

    # Remontando o objeto GeoJSON com todas as páginas
    geojson_data = {
        "type": "FeatureCollection",
        "features": all_features
    }

    # Carregando no GeoDataFrame
    gdf = gpd.GeoDataFrame.from_features(geojson_data)

    # O padrão de saída GeoJSON do ArcGIS é WGS84 (EPSG:4326)
    gdf.set_crs(epsg=4326, allow_override=True, inplace=True)

    # Salvando localmente
    gdf.to_file(output_filename, driver="GPKG")

    print(f"Sucesso! Total de {len(gdf)} registros salvos em: {output_filename}")
    return gdf

# Executar o download (Trocando layer_id para 2, você baixa os polígonos em vez de pontos)
df_cetesb = baixar_cetesb_arcgis(layer_id=1)

# Se quiser conferir o cabeçalho
if df_cetesb is not None:
    display(df_cetesb.head(3))

Iniciando o download da camada 1 via ArcGIS REST API...
Baixados 1000 registros...
Baixados 2000 registros...
Baixados 3000 registros...
Baixados 4000 registros...
Baixados 5000 registros...
Baixados 6000 registros...
Baixados 7000 registros...
Baixados 7152 registros...

Processando os dados no GeoPandas e convertendo para GPKG...
Sucesso! Total de 7152 registros salvos em: areas_contaminadas_cetesb.gpkg


,geometry,OBJECTID,NIS,NumSipol,IdMunSipol,NumSipolText,NomeEmpree,AtividadeSipol,OrigemSipol,MigradoAccess,...,MedidasEngenharia,DesEndereco,DesLocalizacao,DesEndeComple,NomBairro,NomDistrito,CEP,nomUGRHI,Sigla_DG,flag_postos
0,POINT (-46.72948 -23.48958),1,1,484298.0,100.0,100 - 484298,AUTO POSTO PIRITUBANO LTDA,Combustíveis e lubrificantes para veículos; co...,V,V,...,None,"AVENIDA RAIMUNDO PEREIRA DE MAGALHAES, 4520","AVENIDA RAIMUNDO PEREIRA DE MAGALHAES, 4520",None,None,None,51452-00,06 - ALTO TIETÊ,ACRe,1.0
1,POINT (-46.59062 -23.53808),2,2,1005546.0,100.0,100 - 1005546,CONDOMÍNIO LUMINA PARQUE CLUBE (MARSHALL EMPR....,None,V,V,...,None,"R. CONSELHEIRO COTEGIPE, 294",None,None,BELENZINHO,None,3058000,06 - ALTO TIETÊ,AR,0.0
2,POINT (-46.96606 -23.54569),3,3,1005442.0,373.0,373 - 1005442,"PAULISTA S/A COM., PART. E EMPR.",None,V,V,...,None,"EST. ELIAS ALVES DA COSTA, S/N",None,None,PARQUE BOA ESPERA,None,6675200,06 - ALTO TIETÊ,AR,0.0


In [4]:
from google.colab import files
files.download('areas_contaminadas_cetesb.gpkg')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>